In [20]:

import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv('../.env')

data_root = os.getenv('DATA_ROOT')
data_path = os.path.join(data_root, 'ftky-all/dataset_TIST2015_Checkins.txt')
data = pd.read_csv( data_path, sep='\t', encoding='latin1')
data.columns = ['user_id', 'venue_id', 'utc_time', 'timezone_offset']
data.head()


,user_id,venue_id,utc_time,timezone_offset
0,190571,4b4b87b5f964a5204a9f26e3,Tue Apr 03 18:00:07 +0000 2012,180
1,221021,4a85b1b3f964a520eefe1fe3,Tue Apr 03 18:00:08 +0000 2012,-240
2,66981,4b4606f2f964a520751426e3,Tue Apr 03 18:00:08 +0000 2012,-300
3,21010,4c2b4e8a9a559c74832f0de2,Tue Apr 03 18:00:09 +0000 2012,240
4,28761,4b4bade2f964a520cfa326e3,Tue Apr 03 18:00:09 +0000 2012,-240


In [21]:
# tokyo user profile
user_profile_path_tky = os.path.join(data_root, 'ftky-all/dataset_UbiComp2016_UserProfile_TKY.txt')
user_profile_tky = pd.read_csv(user_profile_path_tky, sep='\t')# add header
user_profile_tky.columns = ['user_id', 'gender', 'twitter_friend_count', 'twitter_follower_count']
# gender ratio
user_profile_tky['gender'].value_counts() / len(user_profile_tky)
user_profile_tky.head()



,user_id,gender,twitter_friend_count,twitter_follower_count
0,24195,male,1073,1664
1,45082,male,1961,1272
2,179352,male,2750,2595
3,122942,male,149,975
4,16262,male,5430,8388


In [ ]:
import numpy as np
# user_profile shape
print("Tokyo user profile shape: ", user_profile_tky.shape)
# keep only user_id exists in user_profile
tokyo_data = data[data['user_id'].isin(user_profile_tky['user_id'])]
print("Tokyo data shape: ", tokyo_data.shape)
tokyo_data.head()
# parse utc_time to timestamp
tokyo_data['utc_time'] = pd.to_datetime(tokyo_data['utc_time'], format='%a %b %d %H:%M:%S %z %Y', utc=True)
tokyo_data['timestamp'] = tokyo_data['utc_time'].astype('int64') // 10**9  # Convert to Unix timestamp (seconds)
# Replace utc_time with timestamp and keep only needed columns
# make the offse title to rating, to be consistent with ml-1m
tokyo_data = tokyo_data[['user_id', 'venue_id', 'timezone_offset', 'timestamp']]
# rename offset to rating
tokyo_data.rename(columns={'timezone_offset': 'rating'}, inplace=True)
# rename venue_id to item_id
tokyo_data.rename(columns={'venue_id': 'item_id'}, inplace=True)
# to csv 
tokyo_data.to_csv(f"{data_root}/ftky-tky/data.csv", index=False, header=None)
print("Tokyo unique user_id: ", tokyo_data['user_id'].nunique())
print("Tokyo unique venue_id: ", tokyo_data['item_id'].nunique())


Tokyo user profile shape:  (11873, 4)
Tokyo data shape:  (2145190, 4)


/tmp/ipykernel_4031048/2183957629.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tokyo_data['utc_time'] = pd.to_datetime(tokyo_data['utc_time'], format='%a %b %d %H:%M:%S %z %Y', utc=True)
/tmp/ipykernel_4031048/2183957629.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tokyo_data['timestamp'] = tokyo_data['utc_time'].astype('int64') // 10**9  # Convert to Unix timestamp (seconds)


Tokyo unique user_id:  11873
Tokyo unique venue_id:  298322


In [23]:
tokyo_data

,user_id,item_id,rating,timestamp
5,39350,49bbd6c0f964a520f4531fe3,-240,1333476009
26,20462,4b77034ff964a5207a742ee3,120,1333476024
90,5445,439ec330f964a520102c1fe3,-420,1333476078
286,178657,4ade0f50f964a520e97121e3,60,1333476198
699,7207,4f3ec916e4b065204ad39e44,180,1333476490
...,...,...,...,...
33263588,106,4b56d1eff964a520171c28e3,540,1379373790
33263603,126409,4b79d5cbf964a5202a152fe3,540,1379373811
33263612,38922,4b92005ef964a520ebe233e3,540,1379373829
33263617,328,4bada437f964a520ae603be3,540,1379373838


In [ ]:
import pandas as pd
tokyo_data = pd.read_csv("./dataset/ftky-tky/data.csv", header=None)
tokyo_data


,0,1,2,3
0,39350,49bbd6c0f964a520f4531fe3,1333476009,-240
1,20462,4b77034ff964a5207a742ee3,1333476024,120
2,5445,439ec330f964a520102c1fe3,1333476078,-420
3,178657,4ade0f50f964a520e97121e3,1333476198,60
4,7207,4f3ec916e4b065204ad39e44,1333476490,180
...,...,...,...,...
2145185,106,4b56d1eff964a520171c28e3,1379373790,540
2145186,126409,4b79d5cbf964a5202a152fe3,1379373811,540
2145187,38922,4b92005ef964a520ebe233e3,1379373829,540
2145188,328,4bada437f964a520ae603be3,1379373838,540


In [9]:
# read from
import pandas as pd
tokyo_data = pd.read_csv("./dataset/ftky-tky/data.csv", header=None)
tokyo_data


,0,1,2,3
0,39350,49bbd6c0f964a520f4531fe3,1333476009,-240
1,20462,4b77034ff964a5207a742ee3,1333476024,120
2,5445,439ec330f964a520102c1fe3,1333476078,-420
3,178657,4ade0f50f964a520e97121e3,1333476198,60
4,7207,4f3ec916e4b065204ad39e44,1333476490,180
...,...,...,...,...
2145185,106,4b56d1eff964a520171c28e3,1379373790,540
2145186,126409,4b79d5cbf964a5202a152fe3,1379373811,540
2145187,38922,4b92005ef964a520ebe233e3,1379373829,540
2145188,328,4bada437f964a520ae603be3,1379373838,540
